<a href="https://colab.research.google.com/github/roberthsu2003/machine_learning/blob/main/%E6%A9%9F%E5%99%A8%E5%AD%B8%E7%BF%92%E5%B8%B8%E4%BD%BF%E7%94%A8%E7%9A%84%E5%9C%96%E8%A1%A8/learning_curve_%E5%AD%B8%E7%BF%92%E6%9B%B2%E7%B7%9A.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# 模型診斷：學習曲線 (Learning Curve)

學習曲線是用來分析機器學習模型隨**訓練樣本數量增加**時，其在訓練集與驗證集上的得分（例如 Accuracy、R-squared 或 MSE 損失）變化趨勢。這是有利於診斷以下兩大問題的重要工具：
- **高偏差 (High Bias / Underfitting)**：訓練與驗證得分皆偏低，且兩者非常接近。這代表模型過於簡單，即使增加訓練數據也無法提升表現。
- **高變異 (High Variance / Overfitting)**：訓練得分極高，但驗證得分顯著偏低，兩者間存在巨大差距 (Gap)。這代表模型過度擬合了訓練數據，增加更多數據或降低模型複雜度能有效縮小此差距。

本篇將使用 `Scikit-Learn` 的 `learning_curve` 工具，繪製分類模型的學習曲線。

## 1. 環境設定與下載中文字型

In [ ]:
# 下載中文字型
import os
import wget

if not os.path.exists("ChineseFont.ttf"):
    wget.download("https://github.com/roberthsu2003/machine_learning/raw/refs/heads/main/source_data/ChineseFont.ttf")

In [ ]:
# 設定 Matplotlib 使用下載的中文字型
import matplotlib.pyplot as plt
import matplotlib as mpl
from matplotlib.font_manager import fontManager

fontManager.addfont("ChineseFont.ttf")
mpl.rc('font', family="ChineseFont")
mpl.rcParams['axes.unicode_minus'] = False  # 正常顯示負號

## 2. 載入數據與模型設定
我們使用 Scikit-Learn 內建的手寫數字數據集 (Digits Dataset)，並使用一個決策樹分類器 (Decision Tree Classifier) 來訓練並計算其學習曲線。

In [ ]:
from sklearn.datasets import load_digits
from sklearn.model_selection import learning_curve
from sklearn.tree import DecisionTreeClassifier
import numpy as np

# 載入數據
digits = load_digits()
X, y = digits.data, digits.target

# 設定一個有過擬合傾向的決策樹模型（不限制最大深度）
estimator = DecisionTreeClassifier(max_depth=5, random_state=42)

## 3. 使用 learning_curve 計算得分
`learning_curve` 函數會將數據集分成不同的訓練大小，並利用交叉驗證 (Cross-Validation) 計算訓練與驗證的分數。

In [ ]:
# 計算學習曲線得分
# cv=5 代表 5 折交叉驗證，train_sizes 指定評估的訓練集樣本比例
train_sizes, train_scores, test_scores = learning_curve(
    estimator, X, y, cv=5, train_sizes=np.linspace(0.1, 1.0, 10), scoring='accuracy', n_jobs=-1
)

# 計算多次交叉驗證得分的平均值與標準差
train_mean = np.mean(train_scores, axis=1)
train_std = np.std(train_scores, axis=1)
test_mean = np.mean(test_scores, axis=1)
test_std = np.std(test_scores, axis=1)

## 4. 繪製學習曲線
我們使用 `Matplotlib` 將訓練集得分與驗證集得分隨樣本數增加的曲線畫出來，並以陰影標示得分的標準差區間。

In [ ]:
plt.figure(figsize=(8, 6))

# 繪製訓練得分與陰影區間
plt.plot(train_sizes, train_mean, 'o-', color="r", label="訓練集得分 (Training score)")
plt.fill_between(train_sizes, train_mean - train_std, train_mean + train_std, alpha=0.1, color="r")

# 繪製驗證得分與陰影區間
plt.plot(train_sizes, test_mean, 'o-', color="g", label="驗證集得分 (Cross-validation score)")
plt.fill_between(train_sizes, test_sizes := test_mean - test_std, test_mean + test_std, alpha=0.1, color="g")

# 軸設定
plt.title("決策樹分類器之學習曲線 (Learning Curve)", fontsize=14)
plt.xlabel("訓練樣本數 (Training examples)", fontsize=12)
plt.ylabel("準確度 (Accuracy)", fontsize=12)
plt.legend(loc="best")
plt.grid(True, linestyle="--", alpha=0.5)
plt.ylim(0.4, 1.01)

plt.show()